<a href="https://colab.research.google.com/github/BensonAdomako/Thesis/blob/main/Satellite_Year_Panel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Set up Google Drive
from google.colab import drive
import os
dir = '/content/drive/'
drive.mount(dir)

Mounted at /content/drive/


In [ ]:
import pandas as pd
import numpy as np

# Load Data
plot_df = pd.read_stata("/content/drive/MyDrive/Crop Yield Prediction/LSMS Data/Plotcrop_dataset.dta")

# Yield dataset
yield_df = pd.read_stata('/content/drive/MyDrive/Crop Yield Prediction/LSMS Data/Plot_dataset.dta')

print("plot_df cols:", plot_df.columns.tolist())
print("yield_df cols:", yield_df.columns.tolist())


/tmp/ipython-input-2189321212.py:5: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  plot_df = pd.read_stata("/content/drive/MyDrive/Crop Yield Prediction/LSMS Data/Plotcrop_dataset.dta")


plot_df cols: ['country', 'wave', 'crop_name', 'season', 'pw', 'ea_id_merge', 'ea_id_obs', 'strataid', 'urban', 'admin_1', 'admin_2', 'admin_3', 'admin_1_name', 'admin_2_name', 'admin_3_name', 'lat_modified', 'lon_modified', 'geocoords_id', 'parcel_id_obs', 'parcel_id_merge', 'plot_id_obs', 'plot_id_merge', 'hh_id_obs', 'hh_id_merge', 'harvest_end_month', 'planting_month', 'harvest_kg', 'harvest_value_LCU', 'harvest_value_USD', 'seed_kg', 'seed_value_LCU', 'seed_value_USD', 'improved', 'used_pesticides', 'crop_shock', 'pests_shock', 'rain_shock', 'drought_shock', 'flood_shock']
yield_df cols: ['country', 'wave', 'hh_id_obs', 'hh_id_merge', 'plot_id_obs', 'plot_id_merge', 'parcel_id_obs', 'parcel_id_merge', 'season', 'pw', 'ea_id_merge', 'ea_id_obs', 'strataid', 'admin_1', 'admin_2', 'admin_3', 'admin_1_name', 'admin_2_name', 'lat_modified', 'lon_modified', 'geocoords_id', 'harvest_interview_month', 'planting_interview_month', 'plot_area_GPS', 'farm_size', 'harvest_kg', 'harvest_value_L

/tmp/ipython-input-2189321212.py:8: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  yield_df = pd.read_stata('/content/drive/MyDrive/Crop Yield Prediction/LSMS Data/Plot_dataset.dta')


In [ ]:
merge_keys = ["wave", "season", "ea_id_merge", "parcel_id_merge", "plot_id_merge"]

harvest_merged = plot_df.merge(
    yield_df,
    on=merge_keys,
    how="inner",               # <- only plots that appear in yield_df (maize only)

)

print("Merged shape:", harvest_merged.shape)


Merged shape: (27697, 149)


In [ ]:
print("yield_df cols:", harvest_merged.columns.tolist())

yield_df cols: ['country_x', 'wave', 'crop_name', 'season', 'pw_x', 'ea_id_merge', 'ea_id_obs_x', 'strataid_x', 'urban_x', 'admin_1_x', 'admin_2_x', 'admin_3_x', 'admin_1_name_x', 'admin_2_name_x', 'admin_3_name', 'lat_modified_x', 'lon_modified_x', 'geocoords_id_x', 'parcel_id_obs_x', 'parcel_id_merge', 'plot_id_obs_x', 'plot_id_merge', 'hh_id_obs_x', 'hh_id_merge_x', 'harvest_end_month', 'planting_month', 'harvest_kg_x', 'harvest_value_LCU_x', 'harvest_value_USD_x', 'seed_kg_x', 'seed_value_LCU_x', 'seed_value_USD_x', 'improved_x', 'used_pesticides_x', 'crop_shock_x', 'pests_shock_x', 'rain_shock_x', 'drought_shock_x', 'flood_shock_x', 'country_y', 'hh_id_obs_y', 'hh_id_merge_y', 'plot_id_obs_y', 'parcel_id_obs_y', 'pw_y', 'ea_id_obs_y', 'strataid_y', 'admin_1_y', 'admin_2_y', 'admin_3_y', 'admin_1_name_y', 'admin_2_name_y', 'lat_modified_y', 'lon_modified_y', 'geocoords_id_y', 'harvest_interview_month', 'planting_interview_month', 'plot_area_GPS', 'farm_size', 'harvest_kg_y', 'harve

In [ ]:
# Canonical IDs / location
harvest_merged["country"]      = harvest_merged["country_x"].fillna(harvest_merged["country_y"])
harvest_merged["ea_id_obs"]    = harvest_merged["ea_id_obs_x"].fillna(harvest_merged["ea_id_obs_y"])
harvest_merged["lat_modified"] = harvest_merged["lat_modified_x"].fillna(harvest_merged["lat_modified_y"])
harvest_merged["lon_modified"] = harvest_merged["lon_modified_x"].fillna(harvest_merged["lon_modified_y"])

# Choose your yield variable (from the yield side)
yield_var = "yield_kg"   # you can swap to yield_value_LCU, etc.

keep_cols = [
    # IDs
    "country", "wave", "season",
    "ea_id_merge", "ea_id_obs",
    "lat_modified", "lon_modified",

    # timing
    "harvest_end_month",

    # yield target
    yield_var


]

keep_cols = [c for c in keep_cols if c in harvest_merged.columns]

harvest_df = harvest_merged[keep_cols].copy()

print("harvest_df shape:", harvest_df.shape)
print(harvest_df.head())
#save to csv
harvest_df.to_csv("/content/drive/MyDrive/Crop Yield Prediction/yields_hmonths_df.csv", index=False)


harvest_df shape: (27697, 9)
    country  wave  season      ea_id_merge  ea_id_obs  lat_modified  \
0  Ethiopia   1.0     1.0  010101088801601  1000002.0     14.353816   
1  Ethiopia   1.0     1.0  010101088801601  1000002.0     14.353816   
2  Ethiopia   1.0     1.0  010102088801403  1000004.0     14.288590   
3  Ethiopia   1.0     1.0  010102088801403  1000004.0     14.288590   
4  Ethiopia   1.0     1.0  010103010100106  1000005.0     14.109761   

   lon_modified    harvest_end_month     yield_kg  
0     37.890876  2011-10-01 00:00:00   142.857143  
1     37.890876  2011-11-01 00:00:00  1250.000000  
2     38.210252  2011-10-01 00:00:00  1833.180568  
3     38.210252  2011-10-01 00:00:00  6112.469438  
4     38.473835  2011-12-01 00:00:00    15.950920  


In [ ]:
harvest_df = pd.read_stata('/content/drive/MyDrive/Crop Yield Prediction/LSMS Data/Plotcrop_dataset.dta')

# Convert to datetime (year-month format)
harvest_df['harvest_end_month_dt'] = pd.to_datetime(harvest_df['harvest_end_month'], format='%Y%b', errors='coerce')


/tmp/ipython-input-1095976535.py:1: UnicodeWarning: 
One or more strings in the dta file could not be decoded using utf-8, and
so the fallback encoding of latin-1 is being used.  This can happen when a file
has been incorrectly encoded by Stata or some other software. You should verify
the string values returned are correct.
  harvest_df = pd.read_stata('/content/drive/MyDrive/Crop Yield Prediction/LSMS Data/Plotcrop_dataset.dta')


In [ ]:
import polars as pl
import pandas as pd
import numpy as np

ee_path = "/content/drive/MyDrive/Crop Yield Prediction/EE_combined_long.parquet"

# Load EE combined dataset
df = pl.read_parquet(ee_path)
df = df.to_pandas()

# Drop rows where 'value' is missing
df = df.dropna(subset=['value'])

# Build varname = band_year_month and pivot to wide
df['varname'] = df['band'] + '_' + df['year_month']
df = df.drop_duplicates(subset=['lat_modified', 'lon_modified', 'varname', 'value'])

df = df.pivot(
   index=['lat_modified', 'lon_modified'],
   columns='varname',
   values='value'
).reset_index()

import re
from dateutil.relativedelta import relativedelta

# ----------------------------------------------------------
# Build meta from wide column names in df
# ----------------------------------------------------------
pattern = re.compile(r'^(.*?)_(max|mean|min|sum|std)_(\d{4}_\d{1,2})$')

# Grab all columns that look like they contain a year_month
var_cols = [c for c in df.columns if re.search(r'\d{4}_\d{1,2}', c)]

parsed = []
unmatched = []

for c in var_cols:
    m = pattern.match(c)
    if m:
        variable, stat, year_month = m.groups()
        parsed.append((variable, stat, year_month, c))
    else:
        unmatched.append(c)

if unmatched:
    print("⚠️ Columns that didn’t match expected pattern:")
    for u in unmatched:
        print("   ", u)

meta = pd.DataFrame(parsed, columns=['variable', 'stat', 'year_month', 'column'])

# Clean and validate date suffixes
meta['year_month'] = meta['year_month'].str.strip()

# Fix single-digit months, e.g. 2010_1 → 2010_01
meta['year_month'] = meta['year_month'].str.replace(
    r'(\d{4})_(\d{1})(?!\d)', r'\1_0\2', regex=True
)

meta['year_month_dt'] = pd.to_datetime(
    meta['year_month'], format='%Y_%m', errors='coerce'
)

if meta['year_month_dt'].isna().any():
    bad_cols = meta.loc[meta['year_month_dt'].isna(), 'column'].tolist()
    print("⚠️ Skipping columns with invalid year_month values:")
    for col in bad_cols:
        print("   ", col)

meta = meta.dropna(subset=['year_month_dt'])

print("✅ meta built. Example:")
print(meta.head())



✅ meta built. Example:
  variable stat year_month           column year_month_dt
0      EVI  max    2010_01  EVI_max_2010_01    2010-01-01
1      EVI  max    2010_02  EVI_max_2010_02    2010-02-01
2      EVI  max    2010_03  EVI_max_2010_03    2010-03-01
3      EVI  max    2010_04  EVI_max_2010_04    2010-04-01
4      EVI  max    2010_05  EVI_max_2010_05    2010-05-01


In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree
from dateutil.relativedelta import relativedelta
import re

# Keep valid dates
harvest_df = harvest_df.dropna(subset=['harvest_end_month_dt']).copy()

# Extract month as integer
harvest_df['harvest_month'] = harvest_df['harvest_end_month_dt'].dt.month

# ---- NEW: mode harvest month per point (lat, lon, country) ----
def mode_month(x):
    m = x.mode()
    return m.iloc[0] if not m.empty else np.nan

mode_harvest = (
    harvest_df
    .groupby(['lat_modified', 'lon_modified', 'country'], as_index=False)
    .agg({'harvest_month': mode_month})
    .dropna(subset=['harvest_month'])
)

print("Unique points with a defined mode harvest month:", mode_harvest.shape)


Unique points with a defined mode harvest month: (6925, 4)


In [ ]:
# meta has columns: ['variable', 'stat', 'year_month', 'column', 'year_month_dt']

min_ym = meta['year_month_dt'].min()
max_ym = meta['year_month_dt'].max()

year_min = min_ym.year
year_max = max_ym.year

print("Satellite coverage years:", year_min, "to", year_max)

# Build grid of all years for each point
years = pd.DataFrame({'harvest_year': np.arange(year_min, year_max + 1)})

mode_harvest['key'] = 1
years['key'] = 1

point_years = (
    mode_harvest.merge(years, on='key')
                .drop(columns='key')
)

# Construct synthetic harvest_end_month_dt (day = 1)
point_years['harvest_end_month_dt'] = pd.to_datetime(
    point_years['harvest_year'].astype(str) + "-" +
    point_years['harvest_month'].astype(int).astype(str) + "-01",
    format='%Y-%m-%d'
)

print("Point × year grid:", point_years.shape)
print(point_years.head())

Satellite coverage years: 2010 to 2024
Point × year grid: (103875, 6)
   lat_modified  lon_modified country  harvest_month  harvest_year  \
0      -16.9855     35.249901  Malawi              4          2010   
1      -16.9855     35.249901  Malawi              4          2011   
2      -16.9855     35.249901  Malawi              4          2012   
3      -16.9855     35.249901  Malawi              4          2013   
4      -16.9855     35.249901  Malawi              4          2014   

  harvest_end_month_dt  
0           2010-04-01  
1           2011-04-01  
2           2012-04-01  
3           2013-04-01  
4           2014-04-01  


In [ ]:
# Build KDTree on EE feature grid (df)
tree = cKDTree(df[['lat_modified', 'lon_modified']].values)

# Query nearest neighbor for each point in the point_years grid
dist, idx = tree.query(point_years[['lat_modified', 'lon_modified']].values, k=1)

# tolerance in degrees (same idea as supervisor; adjust if needed)
tol = 1e-4  # ~11m at equator

mask = dist <= tol
print(f"Matched {mask.sum()} of {len(point_years)} point-year rows within {tol}° tolerance.")

# Construct a merged data frame: point_years + EE features
merged_full_df = pd.concat([
    point_years.loc[mask].reset_index(drop=True),
    df.iloc[idx[mask]].reset_index(drop=True).drop(columns=['lat_modified', 'lon_modified'])
], axis=1)

# Add back lat/lon (from point_years)
# (They’re already in point_years.loc[mask], so merged_full_df has them)
print("merged_full_df shape:", merged_full_df.shape)
print(merged_full_df.head(3))


Matched 98280 of 103875 point-year rows within 0.0001° tolerance.
merged_full_df shape: (98280, 1266)
   lat_modified  lon_modified country  harvest_month  harvest_year  \
0      -16.9855     35.249901  Malawi              4          2010   
1      -16.9855     35.249901  Malawi              4          2011   
2      -16.9855     35.249901  Malawi              4          2012   

  harvest_end_month_dt     EVI_max_2010_01     EVI_max_2010_02  \
0           2010-04-01  0.3354778126869772  0.4571863897366428   
1           2011-04-01  0.3354778126869772  0.4571863897366428   
2           2012-04-01  0.3354778126869772  0.4571863897366428   

      EVI_max_2010_03 EVI_max_2010_04  ... temperature_2m_min_mean_2024_03  \
0  0.5350511928394213             NaN  ...               296.8167419785694   
1  0.5350511928394213             NaN  ...               296.8167419785694   
2  0.5350511928394213             NaN  ...               296.8167419785694   

  temperature_2m_min_mean_2024_04 tempe

In [ ]:
# ----------------------------------------------------------
# 1. Parse harvest date (already done above, but just in case)
# ----------------------------------------------------------
merged_full_df['harvest_end_month_dt'] = pd.to_datetime(
    merged_full_df['harvest_end_month_dt']
)

# ----------------------------------------------------------
# 2. Identify feature columns and extract metadata (you already have 'meta')
# ----------------------------------------------------------
# pattern = re.compile(r'^(.*?)_(max|mean|min|sum|std)_(\d{4}_\d{1,2})$')
# var_cols, parsed, meta built as in your existing code

# (Here we assume 'meta' is already defined from df's columns)
# meta: ['variable', 'stat', 'year_month', 'column', 'year_month_dt']

# ----------------------------------------------------------
# 3. Extract 12-month lags for all synthetic harvest years
# ----------------------------------------------------------
records = []

valid_rows = merged_full_df.dropna(subset=['harvest_end_month_dt'])

for _, row in valid_rows.iterrows():
    harvest = row['harvest_end_month_dt']

    subset = meta[
        (meta['year_month_dt'] <= harvest) &
        (meta['year_month_dt'] > harvest - relativedelta(months=12))
    ].copy()

    subset['month_diff'] = (
        (harvest.year - subset['year_month_dt'].dt.year) * 12 +
        (harvest.month - subset['year_month_dt'].dt.month)
    )

    for _, m in subset.iterrows():
        new_name = f"{m['variable']}_{m['stat']}_{m['month_diff']}"
        records.append((
            row['country'],
            row['harvest_year'],           # <- panel year
            row['lat_modified'],
            row['lon_modified'],
            new_name,
            row[m['column']]
        ))

# ----------------------------------------------------------
# 4. Reshape to wide panel: country × harvest_year × point
# ----------------------------------------------------------
df_panel = (
    pd.DataFrame(
        records,
        columns=[
            'country',
            'year',
            'lat_modified',
            'lon_modified',
            'varname',
            'value'
        ]
    )
    .pivot(
        index=['country', 'year', 'lat_modified', 'lon_modified'],
        columns='varname',
        values='value'
    )
    .reset_index()
)

# ----------------------------------------------------------
# 4b. Add harvest_end_month back to the panel index
# ----------------------------------------------------------
# Keep one harvest date per (country, year, lat, lon)
harvest_lookup = (
    point_years.loc[mask, ["country", "harvest_year", "lat_modified", "lon_modified", "harvest_end_month_dt"]]
    .rename(columns={"harvest_year": "year"})
    .drop_duplicates(subset=["country", "year", "lat_modified", "lon_modified"])
)

# Optional: also create a string version (YYYY-MM-01) for easy joins in R
harvest_lookup["harvest_end_month"] = harvest_lookup["harvest_end_month_dt"].dt.strftime("%Y-%m-%d")

# Merge into df_panel
df_panel = df_panel.merge(
    harvest_lookup,
    on=["country", "year", "lat_modified", "lon_modified"],
    how="left"
)

print("✅ Added harvest_end_month_dt to df_panel")
print(df_panel[["country","year","lat_modified","lon_modified","harvest_end_month_dt"]].head())


print("Final full panel shape:", df_panel.shape)
print(df_panel.head(3))

 #save
df_panel.to_csv(
    "/content/drive/MyDrive/Crop Yield Prediction/EE_harvest_ml_full_panel.csv",
    index=False
)
print("✅ Saved full panel with synthetic harvest years.")


✅ Added harvest_end_month_dt to df_panel
    country  year  lat_modified  lon_modified harvest_end_month_dt
0  Ethiopia  2010      3.455701     39.515994           2010-01-01
1  Ethiopia  2010      3.549937     39.184234           2010-01-01
2  Ethiopia  2010      3.982931     38.491368           2010-12-01
3  Ethiopia  2010      4.281844     41.875076           2010-01-01
4  Ethiopia  2010      4.744136     36.045393           2010-01-01
Final full panel shape: (98280, 90)
    country  year  lat_modified  lon_modified            EVI_max_0  \
0  Ethiopia  2010      3.455701     39.515994   0.1607348709810613   
1  Ethiopia  2010      3.549937     39.184234  0.17253798100268572   
2  Ethiopia  2010      3.982931     38.491368  0.13099058350931858   

             EVI_max_1          EVI_max_10          EVI_max_11  \
0                  NaN                 NaN                 NaN   
1                  NaN                 NaN                 NaN   
2  0.21476613555770077  0.1376408592869702

In [ ]:
na_counts = df_panel.isna().sum().sort_values(ascending=False)
print(na_counts.head(20))


GCVI_max_4     9537
EVI_max_4      9537
NDVI_max_4     9537
EVI_max_5      9128
NDVI_max_5     9128
GCVI_max_5     9128
EVI_max_3      8576
NDVI_max_3     8576
GCVI_max_3     8576
GCVI_max_11    8484
NDVI_max_11    8484
EVI_max_11     8484
GCVI_max_6     8281
NDVI_max_6     8281
EVI_max_6      8281
GCVI_max_10    7157
NDVI_max_10    7157
EVI_max_10     7157
GCVI_max_7     6881
NDVI_max_7     6881
dtype: int64


In [ ]:
print("df_panel cols:", df_panel.columns.tolist())

df_panel cols: ['country', 'year', 'lat_modified', 'lon_modified', 'EVI_max_0', 'EVI_max_1', 'EVI_max_10', 'EVI_max_11', 'EVI_max_2', 'EVI_max_3', 'EVI_max_4', 'EVI_max_5', 'EVI_max_6', 'EVI_max_7', 'EVI_max_8', 'EVI_max_9', 'GCVI_max_0', 'GCVI_max_1', 'GCVI_max_10', 'GCVI_max_11', 'GCVI_max_2', 'GCVI_max_3', 'GCVI_max_4', 'GCVI_max_5', 'GCVI_max_6', 'GCVI_max_7', 'GCVI_max_8', 'GCVI_max_9', 'GDD_mean_0', 'GDD_mean_1', 'GDD_mean_10', 'GDD_mean_11', 'GDD_mean_2', 'GDD_mean_3', 'GDD_mean_4', 'GDD_mean_5', 'GDD_mean_6', 'GDD_mean_7', 'GDD_mean_8', 'GDD_mean_9', 'KDD_mean_0', 'KDD_mean_1', 'KDD_mean_10', 'KDD_mean_11', 'KDD_mean_2', 'KDD_mean_3', 'KDD_mean_4', 'KDD_mean_5', 'KDD_mean_6', 'KDD_mean_7', 'KDD_mean_8', 'KDD_mean_9', 'NDVI_max_0', 'NDVI_max_1', 'NDVI_max_10', 'NDVI_max_11', 'NDVI_max_2', 'NDVI_max_3', 'NDVI_max_4', 'NDVI_max_5', 'NDVI_max_6', 'NDVI_max_7', 'NDVI_max_8', 'NDVI_max_9', 'temperature_2m_max_mean_0', 'temperature_2m_max_mean_1', 'temperature_2m_max_mean_10', 'temper

In [ ]:
df_panel.head(25)

,country,year,lat_modified,lon_modified,EVI_max_0,EVI_max_1,EVI_max_10,EVI_max_11,EVI_max_2,EVI_max_3,...,temperature_2m_min_mean_2,temperature_2m_min_mean_3,temperature_2m_min_mean_4,temperature_2m_min_mean_5,temperature_2m_min_mean_6,temperature_2m_min_mean_7,temperature_2m_min_mean_8,temperature_2m_min_mean_9,harvest_end_month_dt,harvest_end_month
0,Ethiopia,2010,3.455701,39.515994,0.1607348709810613,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-01-01,2010-01-01
1,Ethiopia,2010,3.549937,39.184234,0.17253798100268572,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-01-01,2010-01-01
2,Ethiopia,2010,3.982931,38.491368,0.13099058350931858,0.21476613555770077,0.1376408592869702,0.1557176350839331,0.13233666352030496,NaN,...,293.1875482226937,292.4923039939085,291.9040103662428,291.6522260074403,292.6241837954767,293.4297350869967,293.15898773505563,292.8124396677671,2010-12-01,2010-12-01
3,Ethiopia,2010,4.281844,41.875076,0.10241805899521216,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-01-01,2010-01-01
4,Ethiopia,2010,4.744136,36.045393,0.19743689041380957,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-01-01,2010-01-01
5,Ethiopia,2010,4.767689,39.269749,0.14144484861661724,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-01-01,2010-01-01
6,Ethiopia,2010,4.807098,36.081840,0.13289769858424413,0.34369944833879473,0.212982502340388,NaN,0.13055941295894716,NaN,...,296.71101217072345,296.5648382154046,295.9587162564623,296.8458162082482,296.18106731239993,296.76450522878315,296.0521837738573,298.7202342627396,2010-11-01,2010-11-01
7,Ethiopia,2010,4.879055,39.195379,0.15383751894769343,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-01-01,2010-01-01
8,Ethiopia,2010,5.213502,40.035153,0.13410550086990197,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2010-01-01,2010-01-01
9,Ethiopia,2010,5.222998,41.043206,0.12240172511312215,0.358753565470104,NaN,NaN,NaN,NaN,...,294.37411251903166,293.7041977238532,293.20964833504325,295.4571398137613,293.2560531138622,NaN,NaN,NaN,2010-07-01,2010-07-01
